# 11 — HITL feedback round-trip

Demonstrates the full loop: pipeline run → human feedback → feedback applied →
measurably different artifacts. Thin orchestration only; semantics live in
`src/reviewscope_ml/hitl/apply_feedback.py` (its module docstring is the
authoritative description of what each action does on re-run).

Normally feedback comes from the Streamlit app
(`streamlit run src/reviewscope_ml/hitl/app.py`); here we write the same
records programmatically so the notebook runs unattended.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from reviewscope_ml import load_config
from reviewscope_ml.data import load_benchmark
from reviewscope_ml.pipelines import load_run

cfg = load_config(sample_size=1_000, device='cpu')
reviews = load_benchmark(cfg)

# Two-stage run from notebook 10 — its micro→macro hierarchy lets a split
# be answered without re-clustering.
RUN = f'two_stage__{cfg.sample_size}__s42'
art = load_run(cfg.runs_dir / RUN)
print(f'{RUN}: {len(art.clusters)} clusters')

two_stage__1000__s42: 7 clusters


In [2]:
# Before: cluster overview
def overview(a):
    return pd.DataFrame([
        {'cluster': cid, 'label': i.label, 'size': i.size,
         'source': i.label_source,
         'micro_ids': i.micro_cluster_ids,
         'top_terms': ', '.join(w for w, _ in (tuple(t) for t in i.top_terms[:5]))}
        for cid, i in sorted(a.clusters.items())
    ])

before = overview(art)
before

,cluster,label,size,source,micro_ids,top_terms
0,0,room / hotel / great,294,terms_fallback,"[4, 8, 16, 17, 18, 19, 20, 25, 26, 27, 28, 29,...","room, hotel, great, nice, stay"
1,1,hotel / room / stay,79,terms_fallback,"[1, 2, 6]","hotel, room, stay, city, nashville"
2,2,room / hotel / desk,62,terms_fallback,"[41, 42, 43, 44, 45, 46]","room, hotel, desk, told, did"
3,3,oysters / good / food,78,terms_fallback,[0],"oysters, good, food, shrimp, place"
4,4,hotel / quarter / orleans,86,terms_fallback,"[5, 7, 9, 10, 11, 12, 13, 14]","hotel, quarter, orleans, french, great"
5,5,casino / reno / peppermill,98,terms_fallback,[3],"casino, reno, peppermill, room, place"
6,6,beach / hotel / great,40,terms_fallback,"[15, 21, 22, 23, 24]","beach, hotel, great, room, tampa"


In [3]:
# A reviewer session, written as records (the Streamlit app produces exactly these)
from reviewscope_ml.hitl import FeedbackRecord, append_record, session_file

cids = art.cluster_ids
session = session_file(cfg.feedback_dir, RUN)
actions = [
    FeedbackRecord(RUN, 'demo-reviewer', 'rename_label', cluster_id=cids[0],
                   new_label='Front desk & check-in experience'),
    FeedbackRecord(RUN, 'demo-reviewer', 'approve_label', cluster_id=cids[1]),
    FeedbackRecord(RUN, 'demo-reviewer', 'merge_clusters', cluster_id=cids[3],
                   merge_into=cids[2], note='same theme, split by phrasing'),
    FeedbackRecord(RUN, 'demo-reviewer', 'split_cluster', cluster_id=cids[4],
                   note='mixes two distinct complaints'),
]
for r in actions:
    append_record(session, r)
print(f'{len(actions)} records -> {session.name}')

4 records -> two_stage__1000__s42__20260611T125917Z.jsonl


In [4]:
# Apply: produces <run>__reviewed next to the original (original untouched)
from reviewscope_ml.hitl import apply_run_feedback

reviewed_dir = apply_run_feedback(cfg.runs_dir / RUN, cfg.feedback_dir, reviews=reviews)
reviewed = load_run(reviewed_dir)
after = overview(reviewed)
after

,cluster,label,size,source,micro_ids,top_terms
0,0,Front desk & check-in experience,294,hitl_override,"[4, 8, 16, 17, 18, 19, 20, 25, 26, 27, 28, 29,...","room, great, hotel, nice, stay"
1,1,hotel / room / stay,79,hitl_approved,"[1, 2, 6]","hotel, room, stay, city, nashville"
2,2,room / hotel / desk,140,terms_fallback,"[0, 41, 42, 43, 44, 45, 46]","room, hotel, oysters, good, service"
3,5,casino / reno / peppermill,98,terms_fallback,[3],"casino, reno, peppermill, room, place"
4,6,beach / hotel / great,40,terms_fallback,"[15, 21, 22, 23, 24]","beach, hotel, great, tampa, santa"
5,7,hotel / quarter / orleans / part 5,7,split_pending_label,[5],"louis, st, area, hotel, daughter"
6,8,hotel / quarter / orleans / part 7,7,split_pending_label,[7],"ghost, flanagan, beer, tour, bar"
7,9,hotel / quarter / orleans / part 9,5,split_pending_label,[9],"monteleone, carousel, right, floors, hotel"
8,10,hotel / quarter / orleans / part 10,9,split_pending_label,[10],"nola, hotel, great, bar, charm"
9,11,hotel / quarter / orleans / part 11,30,split_pending_label,[11],"orleans, new, hotel, french, bar"


In [5]:
# What changed?
print('applied:', *reviewed.manifest['feedback_applied'], sep='\n  - ')
print()
print(f'clusters before: {len(art.clusters)}  after: {len(reviewed.clusters)}')
print('needs_recluster:', reviewed.manifest.get('needs_recluster'))

applied:
  - merge 3->2
  - split 4 via micro-clusters
  - approve label 1
  - rename 0 -> 'Front desk & check-in experience'

clusters before: 7  after: 13
needs_recluster: []


## What each demonstrated action did

- **rename_label** — label overridden, `label_source='hitl_override'`; later
  LLM passes will not overwrite it.
- **approve_label** — recorded as `hitl_approved`; the report can count
  human-verified labels.
- **merge_clusters** — documents reassigned to the target cluster; term lists
  and word frequencies recomputed from the merged membership.
- **split_cluster** — the macro cluster was decomposed into its micro-clusters
  (promoted to top level). On a flat run the same record flags the cluster in
  `needs_recluster` for targeted re-clustering instead.

The reviewed artifact is a normal run directory — the HITL app, the report
renderer and the future backend consume it exactly like an unreviewed one.